![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 29 -- Lab 2: Flowers with a Pretrained ResNet

An Olympiad-style image task: 4,317 Kaggle photos of five flowers, one folder per class, scored by accuracy. You build the `ImageFolder` pipeline with the ImageNet preprocessing contract, split training from validation, load ResNet-18 with its published weights, freeze the encoder and replace the head with a `Linear(512, 5)`, train the head with Week 3's loop (validation pass and checkpoint), read the accuracy and the confusion matrix, look at the mistakes, and write a submission file. Day 23's practice scenario trained from scratch on the same photos and reached a validation accuracy of 0.68; today you start from ImageNet.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** Kaggle `alxmamaev/flowers-recognition` (about 230 MB), downloaded by `kagglehub` in the setup cell: `flowers/<class>/*.jpg`, 4,317 photos in five folders. You hold out 864 photos (20%) for validation. Use a GPU (Runtime, Change runtime type, T4): the 5 epochs take about 5 minutes there, far longer on a CPU.

---
# Setup

In [ ]:
# kagglehub fetches the Kaggle dataset; Colab has everything else.
!pip install -q kagglehub

In [ ]:
import os, random, time
import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
from sklearn.metrics import confusion_matrix
random.seed(0)
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

In [ ]:
path = kagglehub.dataset_download("alxmamaev/flowers-recognition")
ROOT = os.path.join(path, "flowers")
CLASSES = sorted(os.listdir(ROOT))
print(CLASSES, {c: len(os.listdir(os.path.join(ROOT, c))) for c in CLASSES})

---
# Part 1 -- Data with the contract

The *Preprocessing Contract* and *The Same Data Pipeline and the Same Loop* slides. The published weights expect 224-pixel crops normalised with ImageNet's mean and std; training adds augmentation in front of that, validation does not.

## Task 1: The two transform pipelines

Create `normalize = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])`. Build `train_tf`, a `transforms.Compose` of `RandomResizedCrop(224)`, `RandomHorizontalFlip()`, `ToTensor()` and `normalize`, and `val_tf`, a `Compose` of `Resize(256)`, `CenterCrop(224)`, `ToTensor()` and `normalize`. Apply `val_tf` to one photo of your choice and print the tensor's shape, minimum and maximum.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(t.shape) == (3, 224, 224) and t.min() < -1 and t.max() > 1, 'val_tf: Resize(256), CenterCrop(224), ToTensor, Normalize -> (3, 224, 224) with values below -1 and above 1'
print('Task 1 passed')

## Task 2: ImageFolder, twice

Build `full_train = datasets.ImageFolder(ROOT, transform=train_tf)` and `full_val = datasets.ImageFolder(ROOT, transform=val_tf)`: the same 4,317 files and labels, two different transforms. Print `full_train.classes`, `full_train.class_to_idx` and `len(full_train)`. The folder names became the labels 0 to 4 in alphabetical order.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert full_train.classes == CLASSES and len(full_train) == 4317 and len(full_val) == 4317, 'ImageFolder(ROOT): five classes in alphabetical order, 4,317 photos'
print('Task 2 passed')

## Task 3: Split, DataLoaders, and the contract check

Make `idx = list(range(len(full_train)))`, shuffle it with `random.shuffle` (the seed is set in the setup cell), and take the first 864 indices as `val_idx` and the rest as `train_idx`. Build `train_ds = Subset(full_train, train_idx)` and `val_ds = Subset(full_val, val_idx)`, then `train_loader` (`batch_size=32, shuffle=True, num_workers=2`) and `val_loader` (`batch_size=64, shuffle=False, num_workers=2`). Take one batch `xb, yb = next(iter(train_loader))` and print `xb.shape`, `xb.mean()`, `xb.std()` and `yb[:10]`: the slide's check is a shape of (32, 3, 224, 224), a mean near 0 and a standard deviation near 1.

Syntax hint (the shape of the call, not the answer):

```python
subset = Subset(dataset, list_of_indices)
loader = DataLoader(subset, batch_size=..., shuffle=..., num_workers=2)
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert len(train_ds) == 3453 and len(val_ds) == 864 and tuple(xb.shape) == (32, 3, 224, 224) and abs(xb.mean()) < 1 and 0.5 < xb.std() < 1.6, 'split 3,453 / 864; a batch of (32, 3, 224, 224) with mean near 0 and std near 1'
print('Task 3 passed')

---
# Part 2 -- The model: keep the encoder, teach a new head

The *Mechanism: Swap the Head* and *In Code: The Standard Transfer-Learning Snippet* slides. Load the published weights, freeze everything, replace `fc`.

## Task 4: Load, freeze, replace the head

Create `model = resnet18(weights=ResNet18_Weights.DEFAULT)`. Freeze every parameter with a loop that sets `param.requires_grad = False`, then replace the head: `model.fc = nn.Linear(model.fc.in_features, len(CLASSES))` (a fresh layer is trainable by default). Move the model to `device`. Print the number of trainable parameters, `sum(p.numel() for p in model.parameters() if p.requires_grad)`, and the total.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert n_trainable == 2565 and n_total == 11_179_077, 'frozen encoder + Linear(512, 5): 2,565 trainable of 11,179,077 parameters'
print('Task 4 passed')

## Task 5: The shape habit, and the loss before training

Create `criterion = nn.CrossEntropyLoss()`. Push the batch from Task 3 through the model (`model(xb.to(device))`) under `torch.no_grad()` with the model in `eval()` mode, print the output shape and the loss against `yb.to(device)`. Five classes and a random head: the loss should sit near ln 5 = 1.61, the value of guessing.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert tuple(out.shape) == (32, 5) and 1.0 < loss0 < 2.5, 'the new head gives (32, 5) scores and a loss near ln(5) = 1.61 before training'
print('Task 5 passed')

---
# Part 3 -- Train the head

The *Same Data Pipeline and the Same Loop* slide: Week 3's loop, unchanged. Only the head's 2,565 parameters move, so a step is one forward pass and a tiny update.

## Task 6: The optimizer and an evaluate function

Create `optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)`: only the head's parameters. Write `evaluate(model, loader)` that puts the model in `eval()` mode, loops over the loader under `torch.no_grad()`, sums the loss (`criterion(out, yb).item() * len(yb)`) and the number of correct predictions (`(out.argmax(1) == yb).sum().item()`), and returns the mean loss and the accuracy. Run it on `val_loader` before training: the accuracy of an untrained head is about 0.2.

Syntax hint (the shape of the call, not the answer):

```python
def evaluate(model, loader):
    model.eval()
    with torch.no_grad():
        for xb, yb in loader:
            ...
    return mean_loss, accuracy
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert 0.05 < acc_before < 0.5, 'evaluate before training: the random head scores near chance, 0.2'
print('Task 6 passed')

## Task 7: The loop, written out

Train for 5 epochs. In each epoch: `model.train()`, then for every batch of `train_loader` move `xb, yb` to `device`, compute `loss = criterion(model(xb), yb)`, and do the four moves (`optimizer.zero_grad()`, `loss.backward()`, `optimizer.step()`), adding `loss.item() * len(yb)` to a running total for the epoch's mean. After the batches, call `evaluate(model, val_loader)`, append the three numbers to `history`, print them with the time taken, and when the validation accuracy beats `best_acc`, store it and save the weights with `torch.save(model.state_dict(), "best.pt")`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert best_acc > 0.8 and os.path.exists('best.pt'), 'five epochs of the head alone should pass 0.8 validation accuracy; if not, check the transforms (Normalize) and that the optimizer holds model.fc.parameters()'
print('Task 7 passed')

## Task 8: Read the curves

Plot two panels from `history`: training and validation loss against the epoch on the left, validation accuracy on the right, with titles, axis labels and a legend. Is the validation loss still falling at epoch 5? Compare the accuracy with Day 23's from-scratch scenario (0.68) and with the slide's Oxford Flowers numbers.

In [ ]:
# Your code here


---
# Part 4 -- Score, look, submit

The *Evidence, With Numbers* slide gave other people's numbers; these are yours. Load the best checkpoint, score the validation split, look at the mistakes, write the file.

## Task 9: Best checkpoint, confusion matrix, per-class accuracy

Load the checkpoint back with `model.load_state_dict(torch.load("best.pt", map_location=device))`. Collect the predictions for the whole `val_loader` (`out.argmax(1)`) and the true labels into two lists `y_pred` and `y_true`, print the accuracy, then compute `cm = confusion_matrix(y_true, y_pred)` and show it with `imshow`, the class names on both axes and the count written in every cell. Print the per-class accuracy, `cm.diagonal() / cm.sum(axis=1)`. Which pair of flowers is confused most?

Syntax hint (the shape of the call, not the answer):

```python
cm = confusion_matrix(y_true, y_pred)       # rows: true class, columns: predicted class
ax.text(j, i, cm[i, j], ha="center", va="center")
```

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert cm.sum() == 864 and abs(acc - best_acc) < 0.02, 'the confusion matrix counts all 864 validation photos and its accuracy matches the best epoch'
print('Task 9 passed')

## Task 10: Look at the mistakes

Find the positions where `y_pred != y_true` and show the first 8 of those validation photos with their true and predicted class names as titles. `val_ds[i]` gives the normalised tensor; undo the normalisation for display with `img = t * torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1) + torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)`, then `img.permute(1, 2, 0).clamp(0, 1)`. Are the mistakes the pairs the confusion matrix pointed at?

In [ ]:
# Your code here


## Task 11: Write the submission

Build a DataFrame with one row per validation photo: `image`, the file name (`os.path.basename(full_val.samples[i][0])` for each index `i` in `val_idx`, in the loader's order), and `label`, the predicted class name. Save it as `submission.csv` without the index, read it back, and print its shape and first five rows. Every label must be one of the five class names.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
check = pd.read_csv('submission.csv')
assert list(check.columns) == ['image', 'label'] and len(check) == 864 and set(check['label']) <= set(CLASSES), 'submission.csv: 864 rows, columns image and label, labels among the five class names'
print('Task 11 passed')

---
## Task 12: Reflect

Only 2,565 of 11,179,077 parameters were trained, on 3,453 photos; Day 23's practice scenario trained about 288,000 from scratch on the same photos and memorised them. Explain the difference with the slide's photos-per-parameter argument. Task 5 printed a loss near 1.61 before training: where does that number come from? Which two flowers did your confusion matrix confuse most, and what in the photos of Task 10 explains it? Tomorrow's lecture unfreezes layers: which layer would you unfreeze first, with which learning rate, and what would you watch to see whether it helped or hurt? Finally, in a contest the hidden test photos come in one folder without class subfolders: what changes in your pipeline, and what stays the same?

*Your answers here*

---
## Conclusion

- **The contract in the pipeline**: `ImageFolder` with `Resize`, `CenterCrop` and the ImageNet `Normalize` for validation, `RandomResizedCrop` and flips in front of it for training; a batch checked for shape, mean and std.
- **Keep the encoder, teach a new head**: ResNet-18 loaded with its published weights, every parameter frozen, `fc` replaced by `Linear(512, 5)`: 2,565 trainable parameters.
- **Week 3's loop, unchanged**: five epochs with a validation pass and a checkpoint of the best epoch, the loss starting at ln 5 and the accuracy far above the from-scratch scenario.
- **Score, look, submit**: the confusion matrix and per-class accuracy, the misclassified photos shown, and a submission with one row per photo.

---

Made By **Sattam Altwaim**